# ⚽ AI Football Agency Strategy Desk — SQLite Vector DB & 공식 FIFA PDF RAG 검증 노트북

이 노트북은 **`src/sports_agency/rag/sqlite_vector_store.py`** 에 구현된 이원화 SQLite Vector DB(`data/agency_desk.db`) 및 공식 FIFA 법률 PDF RAG 파이프라인을 단계별로 검증하고 표(`DataFrame`)로 한눈에 확인할 수 있도록 구성되어 있습니다.

### 📌 핵심 설정값
- **Text Splitter**: `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)`
- **기본 검색 개수 (`top_k`)**: `5`
- **법률·규정집 임베딩 모델 (`rag_regulations_vec`)**: `gemini-embedding-2` (`768`차원, 공식 FIFA PDF 4종 62페이지 + `regulations.md`)
- **전술·스카우팅 노트 임베딩 모델 (`scouting_notes_vec`)**: `gemini-embedding-001` (`768`차원, 유럽 5대 리그 96개 구단 & 선수단)

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sports_agency.config import CONFIG
from sports_agency.rag.sqlite_vector_store import VECTOR_STORE

pd.set_option("display.max_colwidth", 140)

config_df = pd.DataFrame([
    {"항목": "RAG Chunk Size", "설정값": CONFIG.rag_chunk_size},
    {"항목": "RAG Chunk Overlap", "설정값": CONFIG.rag_chunk_overlap},
    {"항목": "RAG Default Top-K", "설정값": CONFIG.rag_default_top_k},
    {"항목": "법률/규정 임베딩 모델 (Primary)", "설정값": f"{CONFIG.primary_embedding_model} ({CONFIG.embedding_dimension}d)"},
    {"항목": "전술/스카우팅 임베딩 모델 (Light)", "설정값": f"{CONFIG.light_embedding_model} ({CONFIG.embedding_dimension}d)"},
    {"항목": "SQLite Vector DB 경로", "설정값": str(CONFIG.db_path)},
])
display(Markdown("### 1️⃣ RAG 시스템 설정 확인"))
display(config_df)

### 1️⃣ RAG 시스템 설정 확인

,항목,설정값
0,RAG Chunk Size,800
1,RAG Chunk Overlap,150
2,RAG Default Top-K,5
3,법률/규정 임베딩 모델 (Primary),gemini-embedding-2 (768d)
4,전술/스카우팅 임베딩 모델 (Light),gemini-embedding-001 (768d)
5,SQLite Vector DB 경로,C:\Users\DJ\Documents\ai_google\sports_agency\data\agency_desk.db


## 2️⃣ 공식 FIFA 법률 PDF 4종 + `regulations.md` 청킹(`chunk_size=800`, `chunk_overlap=150`) 검증

In [2]:
chunks = VECTOR_STORE.build_all_regulation_chunks()
chunks_df = pd.DataFrame(chunks)
chunks_df["char_length"] = chunks_df["content"].str.len()
chunks_df["source_type"] = chunks_df["chunk_id"].apply(lambda x: "Official FIFA PDF" if x.startswith("pdf_") else "Markdown Guide")

summary_df = chunks_df.groupby(["source_type", "doc_category"]).agg(
    청크_수=("chunk_id", "count"),
    평균_글자수=("char_length", lambda s: int(s.mean())),
    최소_글자수=("char_length", "min"),
    최대_글자수=("char_length", "max"),
).reset_index()

display(Markdown(f"**총 생성된 법률·규정 청크 수: `{len(chunks_df)}`개** (모든 청크 최대 길이 `<= {CONFIG.rag_chunk_size}`자 검증 통과)"))
display(summary_df)

display(Markdown("#### 📄 청크 샘플 미리보기 (상위 8개)"))
display(chunks_df[["chunk_id", "source_type", "title", "source_citation", "char_length", "content"]].head(8))

**총 생성된 법률·규정 청크 수: `687`개** (모든 청크 최대 길이 `<= 800`자 검증 통과)

,source_type,doc_category,청크_수,평균_글자수,최소_글자수,최대_글자수
0,Markdown Guide,FIFA_FFAR_FEE,2,617,600,635
1,Markdown Guide,GBE_WORK_PERMIT,3,631,546,797
2,Markdown Guide,NON_EU_QUOTA,1,216,216,216
3,Markdown Guide,PSR_CONTRACT_LAW,2,517,381,653
4,Official FIFA PDF,FIFA_FFAR_FEE,308,667,62,800
5,Official FIFA PDF,PSR_CONTRACT_LAW,371,656,42,800


#### 📄 청크 샘플 미리보기 (상위 8개)

,chunk_id,source_type,title,source_citation,char_length,content
0,md_doc_1_chunk_1,Markdown Guide,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 1),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),550,## [문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준\n- **출처**: The FA Men's Players Point-Based System Guidelines (EPL ...
1,md_doc_1_chunk_2,Markdown Guide,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 2),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),546,"### 2. 소속 리그 등급 (League Band) 및 국내 리그 출전 시간 점수\n- **Band 1 (유럽 5대 리그: EPL, 라리가, 분데스리가, 세리에A, 리그1)**: 기본 12점 (출전 시간 30% 이상 시 10~12점)\n- *..."
2,md_doc_2_chunk_1,Markdown Guide,[문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정 (Part 1),UEFA National Associations Player Registration Rules Summary,797,## [문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정\n- **출처**: UEFA National Associations Player Registration Rules Summary\n- **기준일**: 2025-...
3,md_doc_2_chunk_2,Markdown Guide,[문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정 (Part 2),UEFA National Associations Player Registration Rules Summary,216,### 4. 이탈리아 (Serie A)\n- 기존 보유 Non-EU 선수 외에 **한 시즌당 해외 리그에서 새로 영입할 수 있는 Non-EU 선수는 최대 2명**(기존 Non-EU 선수 방출/만료 조건 포함)으로 엄격히 제한됩니다.\n\n###...
4,md_doc_3_chunk_1,Markdown Guide,[문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황 (Part 1),FIFA Football Agent Regulations (FFAR Article 15) & FIFA Circular No. 1873 (Worldwide Temporary Suspension Notice),635,## [문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황\n- **출처**: FIFA Football Agent Regulations (FFAR Article 15) & FIFA ...
5,md_doc_3_chunk_2,Markdown Guide,[문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황 (Part 2),FIFA Football Agent Regulations (FFAR Article 15) & FIFA Circular No. 1873 (Worldwide Temporary Suspension Notice),600,"### 2. [중요 실무 주의사항] EU 사법재판소 심사 및 국가별 효력 정지(Suspension) 현황\n- 유럽 사법재판소(CJEU) 심리와 독일(도르트문트 법원 가처분), 영국(잉글랜드 FA 중재 판정, 2023년 11월) 등의 반발로 인..."
6,md_doc_4_chunk_1,Markdown Guide,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 1),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),653,## [문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드\n- **출처**: Premier League Handbook (Profitability and Sustainability Rules...
7,md_doc_4_chunk_2,Markdown Guide,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 2),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),381,### 2. 바이아웃(Release Clause) 및 초상권(Image Rights) 표준 조항\n- **바이아웃(Release Clause)**: 스페인 라리가(Royal Decree 1006/1985)는 모든 프로 선수 계약서에 의무적으로 ...


## 3️⃣ SQLite Vector DB (`data/agency_desk.db`) 인덱싱 상태 확인

In [3]:
idx_status = VECTOR_STORE.ensure_indexed(force_reindex=False)
if idx_status["rag_regulations_chunks"] != len(chunks_df):
    idx_status = VECTOR_STORE.ensure_indexed(reindex_regulations_only=True)

display(pd.DataFrame([idx_status]).T.rename(columns={0: "상태 값"}))

,상태 값
sqlite_db_path,C:\Users\DJ\Documents\ai_google\sports_agency\data\agency_desk.db
sqlite_vec_extension,True
chunk_size,800
chunk_overlap,150
default_top_k,5
rag_regulations_chunks,687
rag_regulations_embedding_model,gemini-embedding-2
scouting_notes_chunks,3260
scouting_notes_embedding_model,gemini-embedding-001


## 4️⃣ 법률 & 공식 FIFA PDF 벡터 검색 테스트 (`gemini-embedding-2`, `top_k=5`)
실무에서 가장 자주 발생하는 4가지 핵심 질의(수수료 유예 Circular 1873, 영국 GBE 워크퍼밋, 5대 리그 Non-EU 쿼터, FIFA FFAR 제15조 원문)를 검색하여 상위 `top_k=5` 청크를 표로 확인합니다.

In [4]:
test_queries = [
    "FIFA Circular 1873 worldwide temporary suspension of FFAR service fee cap Article 15",
    "영국 FA GBE 워크퍼밋 15점 자동 통과 기준 및 A매치 출전 비율",
    "스페인 라리가(La Liga) 및 이탈리아 세리에A 비유럽(Non-EU) 외국인 선수 등록 쿼터 제한",
    "What is the service fee cap for a Football Agent representing an Individual or Engaging Entity under FFAR?",
]

for idx, q in enumerate(test_queries, start=1):
    results = VECTOR_STORE.search_regulations(query=q, top_k=CONFIG.rag_default_top_k)
    res_df = pd.DataFrame(results)[[
        "similarity_score", "doc_category", "title", "source_citation", "effective_date", "char_length", "content"
    ]]
    res_df.index = [f"Top-{i}" for i in range(1, len(res_df) + 1)]
    display(Markdown(f"### 🔍 질의 {idx}: `{q}` (반환 개수: `top_k={len(res_df)}`)"))
    display(res_df)

### 🔍 질의 1: `FIFA Circular 1873 worldwide temporary suspension of FFAR service fee cap Article 15` (반환 개수: `top_k=5`)

,similarity_score,doc_category,title,source_citation,effective_date,char_length,content
Top-1,1.0859,FIFA_FFAR_FEE,FIFA Circular No. 1873 — FFAR Worldwide Temporary Suspension Update (p.2),"FIFA Circular no. 1873 (Zurich, 30 December 2023 Official PDF, 3 Pages), Page 2",2023년 12월 30일 발효 ~ 2026년 현재 유효,787,any transfer which has a link to the European Union. Implementing the Injunction only for \ntransfers linked to the European Union would...
Top-2,1.0767,FIFA_FFAR_FEE,FIFA Circular No. 1873 — FFAR Worldwide Temporary Suspension Update (p.2),"FIFA Circular no. 1873 (Zurich, 30 December 2023 Official PDF, 3 Pages), Page 2",2023년 12월 30일 발효 ~ 2026년 현재 유효,793,"German court decision, until the European Court of Justice renders a final decision in the \npending procedures concerning the FFAR, \n..."
Top-3,1.0741,FIFA_FFAR_FEE,FIFA Circular No. 1873 — FFAR Worldwide Temporary Suspension Update (p.2),"FIFA Circular no. 1873 (Zurich, 30 December 2023 Official PDF, 3 Pages), Page 2",2023년 12월 30일 발효 ~ 2026년 현재 유효,757,2 \n \n• The submission rule (article 4 paragraph 2; article 16 paragraph 2 b); article 3 paragraphs \n2 c) and d); article 20; and arti...
Top-4,1.0715,FIFA_FFAR_FEE,FIFA Circular No. 1873 — FFAR Worldwide Temporary Suspension Update (p.1),"FIFA Circular no. 1873 (Zurich, 30 December 2023 Official PDF, 3 Pages), Page 1",2023년 12월 30일 발효 ~ 2026년 현재 유효,798,"the FFAR, is pending before the European Court of Justice. \n \nDistrict Court of Dortmund (Landgericht Dortmund) case 8 O 1/23 \n \nOn ..."
Top-5,1.0645,FIFA_FFAR_FEE,FIFA Circular No. 1873 — FFAR Worldwide Temporary Suspension Update (p.1),"FIFA Circular no. 1873 (Zurich, 30 December 2023 Official PDF, 3 Pages), Page 1",2023년 12월 30일 발효 ~ 2026년 현재 유효,206,"• The prohibition of double representation (article 12 paragraphs 8-10) \n• The reporting obligations (article 16 paragraphs 2 h), j), k..."


### 🔍 질의 2: `영국 FA GBE 워크퍼밋 15점 자동 통과 기준 및 A매치 출전 비율` (반환 개수: `top_k=5`)

,similarity_score,doc_category,title,source_citation,effective_date,char_length,content
Top-1,1.1061,GBE_WORK_PERMIT,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 1),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),2024-2026 시즌 적용 기준,550,## [문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준\n- **출처**: The FA Men's Players Point-Based System Guidelines (EPL ...
Top-2,1.0624,GBE_WORK_PERMIT,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 2),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),2024-2026 시즌 적용 기준,546,"### 2. 소속 리그 등급 (League Band) 및 국내 리그 출전 시간 점수\n- **Band 1 (유럽 5대 리그: EPL, 라리가, 분데스리가, 세리에A, 리그1)**: 기본 12점 (출전 시간 30% 이상 시 10~12점)\n- *..."
Top-3,0.9565,GBE_WORK_PERMIT,[문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정 (Part 1),UEFA National Associations Player Registration Rules Summary,2025-2026 시즌 기준,797,## [문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정\n- **출처**: UEFA National Associations Player Registration Rules Summary\n- **기준일**: 2025-...
Top-4,0.9152,PSR_CONTRACT_LAW,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 1),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),2024-2026 시즌 회계연도 적용 기준,653,## [문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드\n- **출처**: Premier League Handbook (Profitability and Sustainability Rules...
Top-5,0.7860,FIFA_FFAR_FEE,[문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황 (Part 2),FIFA Football Agent Regulations (FFAR Article 15) & FIFA Circular No. 1873 (Worldwide Temporary Suspension Notice),2023년 12월 30일 유예 공문 발효 ~ 2026년 현재 소송 및 국가별 차등 적용 중,600,"### 2. [중요 실무 주의사항] EU 사법재판소 심사 및 국가별 효력 정지(Suspension) 현황\n- 유럽 사법재판소(CJEU) 심리와 독일(도르트문트 법원 가처분), 영국(잉글랜드 FA 중재 판정, 2023년 11월) 등의 반발로 인..."


### 🔍 질의 3: `스페인 라리가(La Liga) 및 이탈리아 세리에A 비유럽(Non-EU) 외국인 선수 등록 쿼터 제한` (반환 개수: `top_k=5`)

,similarity_score,doc_category,title,source_citation,effective_date,char_length,content
Top-1,1.0733,GBE_WORK_PERMIT,[문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정 (Part 1),UEFA National Associations Player Registration Rules Summary,2025-2026 시즌 기준,797,## [문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정\n- **출처**: UEFA National Associations Player Registration Rules Summary\n- **기준일**: 2025-...
Top-2,1.0661,NON_EU_QUOTA,[문서 2] 유럽 5대 리그 비EU(Non-EU) 외국인 선수 등록 쿼터 규정 (Part 2),UEFA National Associations Player Registration Rules Summary,2025-2026 시즌 기준,216,### 4. 이탈리아 (Serie A)\n- 기존 보유 Non-EU 선수 외에 **한 시즌당 해외 리그에서 새로 영입할 수 있는 Non-EU 선수는 최대 2명**(기존 Non-EU 선수 방출/만료 조건 포함)으로 엄격히 제한됩니다.\n\n###...
Top-3,0.7981,PSR_CONTRACT_LAW,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 2),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),2024-2026 시즌 회계연도 적용 기준,381,### 2. 바이아웃(Release Clause) 및 초상권(Image Rights) 표준 조항\n- **바이아웃(Release Clause)**: 스페인 라리가(Royal Decree 1006/1985)는 모든 프로 선수 계약서에 의무적으로 ...
Top-4,0.7954,GBE_WORK_PERMIT,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 2),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),2024-2026 시즌 적용 기준,546,"### 2. 소속 리그 등급 (League Band) 및 국내 리그 출전 시간 점수\n- **Band 1 (유럽 5대 리그: EPL, 라리가, 분데스리가, 세리에A, 리그1)**: 기본 12점 (출전 시간 30% 이상 시 10~12점)\n- *..."
Top-5,0.7541,PSR_CONTRACT_LAW,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 1),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),2024-2026 시즌 회계연도 적용 기준,653,## [문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드\n- **출처**: Premier League Handbook (Profitability and Sustainability Rules...


### 🔍 질의 4: `What is the service fee cap for a Football Agent representing an Individual or Engaging Entity under FFAR?` (반환 개수: `top_k=5`)

,similarity_score,doc_category,title,source_citation,effective_date,char_length,content
Top-1,1.1025,FIFA_FFAR_FEE,FIFA Football Agent Regulations (FFAR Official Full Text PDF) (p.21),"FIFA Football Agent Regulations (Official FIFA PDF, 39 Pages), Page 21",2023년 10월 1일 발효 (Circular 1873 일부 유예 병행),786,21\n ACTING AS A FOOTBALL AGENT\nIII.\nArticle 15: Service fee cap\n1. The service fee payable to a Football Agent for the performan...
Top-2,1.0941,FIFA_FFAR_FEE,Explanatory Notes on the FIFA Football Agent Regulations (Enclosure 1) (p.4),"FIFA FFAR Explanatory Notes (Official PDF Enclosure 1, 10 Pages), Page 4",2023-2026 시즌 기준,771,"any conditional payments) is less than USD 200,000 (or equivalent) and where certain other \nconditions are met. \n \nA football agent m..."
Top-3,1.0873,FIFA_FFAR_FEE,Explanatory Notes on the FIFA Football Agent Regulations (Enclosure 1) (p.5),"FIFA FFAR Explanatory Notes (Official PDF Enclosure 1, 10 Pages), Page 5",2023-2026 시즌 기준,737,Explanatory notes on the FIFA Football Agent Regulations – January 2023 4 \nClient \nService Fee Cap \nIndividual’s annual \nre...
Top-4,1.0844,FIFA_FFAR_FEE,FIFA Football Agent Regulations Official FAQs (March 2023 Edition) (p.32),"FIFA Football Agent Regulations FAQs (Official FIFA PDF, 56 Pages), Page 32",2023-2026 시즌 기준,758,Engaging Entity \nand Individual \n(permitted dual \nrepresentation)\n10% of the Individual’s \nRemuneration\n6% of the Individual’s \n...
Top-5,1.0833,FIFA_FFAR_FEE,FIFA Football Agent Regulations Official FAQs (March 2023 Edition) (p.32),"FIFA Football Agent Regulations FAQs (Official FIFA PDF, 56 Pages), Page 32",2023-2026 시즌 기준,777,32\n4.20 What is the maximum service fee payable for the provision of \nFootball Agent Services in a Transaction?\nRegardless of the nu...


## 5️⃣ 구단 전술 적합도(Tactical Fit) 벡터 검색 테스트 (`gemini-embedding-001`, `top_k=5`)
선수의 포지션 및 전술 강점 텍스트와 96개 구단의 포지션별 영입 니즈(`scouting_notes_vec`) 간의 코사인 유사도를 계산해 상위 5개 구단을 추출합니다.

In [5]:
tactical_query = "Right Winger (left-footed) | 빠른 역습 전개, 90분당 전진 패스 및 박스 타격 능력 우수"
tactical_results = VECTOR_STORE.search_tactical_fit(player_strengths=tactical_query, position="RW", top_k=CONFIG.rag_default_top_k)
tac_df = pd.DataFrame(tactical_results)[[
    "tactical_similarity_score", "club_name", "league", "position", "embedding_model", "tactical_note"
]]
tac_df.index = [f"Top-{i}" for i in range(1, len(tac_df) + 1)]
display(Markdown(f"### 🎯 전술 질의: `{tactical_query}` (포지션: `RW`, `top_k={len(tac_df)}`)"))
display(tac_df)

### 🎯 전술 질의: `Right Winger (left-footed) | 빠른 역습 전개, 90분당 전진 패스 및 박스 타격 능력 우수` (포지션: `RW`, `top_k=5`)

,tactical_similarity_score,club_name,league,position,embedding_model,tactical_note
Top-1,0.7063,Parma Calcio 1913,Serie A,RW,"gemini-embedding-001 (Vertex AI ADC, 768d)",[Parma Calcio 1913 - RW] 전술 스타일: Serie A Tactical Profile (Carlos Cuesta) | 영입 니즈: Parma Calcio 1913 (Serie A): 포지션 뎁스 부족(전문 자원 1명)로 인한 ...
Top-2,0.6940,Real Oviedo,La Liga,RW,"gemini-embedding-001 (Vertex AI ADC, 768d)",[Real Oviedo - RW] 전술 스타일: La Liga Tactical Profile (Veljko Paunovic) | 영입 니즈: Real Oviedo (La Liga): 포지션 뎁스 부족(전문 자원 1명)로 인한 RW 포지션 타겟 ...
Top-3,0.6892,TSG 1899 Hoffenheim,Bundesliga,RW,"gemini-embedding-001 (Vertex AI ADC, 768d)",[TSG 1899 Hoffenheim - RW] 전술 스타일: Bundesliga Tactical Profile (Christian Ilzer) | 영입 니즈: TSG 1899 Hoffenheim 스쿼드 내 전문 RW 주전 자원이 부족하여 즉시...
Top-4,0.6884,Udinese Calcio,Serie A,RW,"gemini-embedding-001 (Vertex AI ADC, 768d)",[Udinese Calcio - RW] 전술 스타일: Serie A Tactical Profile (Kosta Runjaić) | 영입 니즈: Udinese Calcio (Serie A): 포지션 뎁스 부족(전문 자원 1명)로 인한 RW 포지션...
Top-5,0.6837,Nottingham Forest,Premier League,RW,"gemini-embedding-001 (Vertex AI ADC, 768d)",[Nottingham Forest - RW] 전술 스타일: Premier League Tactical Profile (Nuno Espírito Santo) | 영입 니즈: Nottingham Forest (Premier League): 포지션 ...


## 6️⃣ [자유 실험 셀] 원하는 검색어로 직접 RAG 테스트하기
아래 `custom_query` 값을 바꾸고 셀을 실행(`Shift + Enter`)하면 실시간으로 `gemini-embedding-2` 벡터 검색 결과를 표로 확인할 수 있습니다.

In [6]:
custom_query = "프리미어리그 PSR 이적료 감가상각 5년 상한 및 출전 보너스 전환 협상 전략"
custom_results = VECTOR_STORE.search_regulations(query=custom_query, top_k=5)
custom_df = pd.DataFrame(custom_results)[[
    "similarity_score", "title", "source_citation", "effective_date", "char_length", "content"
]]
custom_df.index = [f"Top-{i}" for i in range(1, len(custom_df) + 1)]
display(custom_df)

,similarity_score,title,source_citation,effective_date,char_length,content
Top-1,1.0946,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 1),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),2024-2026 시즌 회계연도 적용 기준,653,## [문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드\n- **출처**: Premier League Handbook (Profitability and Sustainability Rules...
Top-2,1.0101,[문서 4] 프리미어리그 PSR / UEFA FSR 재정 규정(감가상각) 및 선수 표준 계약 특약 가이드 (Part 2),Premier League Handbook (Profitability and Sustainability Rules - PSR) & UEFA Financial Sustainability Regulations (Article 85-92),2024-2026 시즌 회계연도 적용 기준,381,### 2. 바이아웃(Release Clause) 및 초상권(Image Rights) 표준 조항\n- **바이아웃(Release Clause)**: 스페인 라리가(Royal Decree 1006/1985)는 모든 프로 선수 계약서에 의무적으로 ...
Top-3,0.8851,[문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황 (Part 2),FIFA Football Agent Regulations (FFAR Article 15) & FIFA Circular No. 1873 (Worldwide Temporary Suspension Notice),2023년 12월 30일 유예 공문 발효 ~ 2026년 현재 소송 및 국가별 차등 적용 중,600,"### 2. [중요 실무 주의사항] EU 사법재판소 심사 및 국가별 효력 정지(Suspension) 현황\n- 유럽 사법재판소(CJEU) 심리와 독일(도르트문트 법원 가처분), 영국(잉글랜드 FA 중재 판정, 2023년 11월) 등의 반발로 인..."
Top-4,0.8228,[문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황 (Part 1),FIFA Football Agent Regulations (FFAR Article 15) & FIFA Circular No. 1873 (Worldwide Temporary Suspension Notice),2023년 12월 30일 유예 공문 발효 ~ 2026년 현재 소송 및 국가별 차등 적용 중,635,## [문서 3] FIFA 축구 에이전트 규정 (FFAR) 제15조 수수료 상한 및 최신 유예(Suspension) 현황\n- **출처**: FIFA Football Agent Regulations (FFAR Article 15) & FIFA ...
Top-5,0.7958,[문서 1] 영국 축구협회(FA) GBE (Governing Body Endorsement) 워크퍼밋 점수 산정 기준 (Part 2),The FA Men's Players Point-Based System Guidelines (EPL & EFL Championship),2024-2026 시즌 적용 기준,546,"### 2. 소속 리그 등급 (League Band) 및 국내 리그 출전 시간 점수\n- **Band 1 (유럽 5대 리그: EPL, 라리가, 분데스리가, 세리에A, 리그1)**: 기본 12점 (출전 시간 30% 이상 시 10~12점)\n- *..."
